In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions
from textblob import TextBlob

## **Upload the document**

In [61]:
data=open("data.txt",encoding="utf-8").read()

## convert to lower case

In [62]:
data=data.lower()

## remove extra spaces

In [63]:
data=re.sub(r'\s{2,}',"",data)

## removing numbers like 1.,2.....

In [64]:
data=re.sub(r'\d+\.','',data)

## remove contractions

In [65]:
data=contractions.fix(data) # if string type of data directly upload


## remove puntuations and spl characters

In [66]:
data=re.sub(r'[^0-9A-za-z\s]','',data)
data

'machine learning ml is a branch of artificial intelligence ai that enables computers to learn from data and improve their performance without being explicitly programmed for every task instead of following fixed instructions machine learning systems identify patterns relationships and trends in historical data to make predictions or decisions on new unseen data it combines concepts from mathematics statistics computer science and data analysis to build intelligent models capable of solving realworld problems machine learning is widely used in applications such as recommendation systems fraud detection medical diagnosis selfdriving cars speech recognition spam filtering and predictive analytics artificial intelligence is the broad field focused on creating machines that can perform tasks requiring human intelligence such as reasoning learning decisionmaking and problemsolving machine learning is a subset of artificial intelligence that focuses specifically on enabling systems to learn 

## spell correction--> TEXTBLOB

In [67]:
# data=TextBlob(data).correct()
# data

## Tokenization

## Spacy and lemmatization

In [68]:
import spacy
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=" ".join(updated_tokens).strip()
data


'machine learning ml branch artificial intelligence ai enable computer learn datum improve performance explicitly program task instead follow fix instruction machine learn system identify pattern relationship trend historical datum prediction decision new unseen datum combine concept mathematic statistics computer science datum analysis build intelligent model capable solve realworld problem machine learning widely application recommendation system fraud detection medical diagnosis selfdrive car speech recognition spam filtering predictive analytic artificial intelligence broad field focus create machine perform task require human intelligence reasoning learning decisionmake problemsolve machine learning subset artificial intelligence focus specifically enable system learn datum deep learning specialized subset machine learning use artificial neural network multiple hide layer automatically learn complex feature large amount data datum science interdisciplinary field combine statistic 

## chunking( converting doc-> chunks)

In [69]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
    
)
chunks=splitter.split_text(data)

## embeddings( converting chunks to vectors)

In [72]:
embedding_model = SentenceTransformer(
    model_name_or_path="sentence-transformers/all-MiniLM-L6-v2"
)
chunk_embeddings = embedding_model.encode(chunks).astype("float32")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2587.00it/s]


In [73]:
dimension=chunk_embeddings.shape[1]
dimension

384

In [74]:
faiss.normalize_L2(chunk_embeddings)

In [75]:
index_faiss_db=faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embeddings)


In [83]:
def rag_query(query, k=2):
    query = re.sub(r'[^0-9a-zA-Z\s]', '', query).strip()
    if not query:
        raise ValueError("Query is empty after removing punctuation.")
    query_embedding = embedding_model.encode(query).astype("float32")
    if query_embedding.ndim == 1:
        query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    k = min(k, index_faiss_db.ntotal)
    distances, indices = index_faiss_db.search(query_embedding, k)
    for idx in indices[0]:
        if idx >= 0:
            print(idx)
    return indices, distances

user_input = "Explain Machine Learning?"
response = rag_query(user_input)
print(response)

213
226
(array([[213, 226]]), array([[0.60080856, 0.5305493 ]], dtype=float32))


In [ ]:
def rag_query(query, k=2):
    query = re.sub(r'[^0-9a-zA-Z\s]', '', query).strip()
    if not query:
        raise ValueError("Query is empty after removing punctuation.")

    query_embedding = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    k = min(k, index_faiss_db.ntotal)
    distances, indices = index_faiss_db.search(query_embedding, k)

    for idx in indices[0]: 
        if idx >= 0:
            print(idx)

    R_chunks = [chunks[i] for i in indices[0] if i >= 0]
    R_str = " ".join(R_chunks)

    prompt = (
            "You are a helpful assistant\n"
            "Assigned Task for you : Structure my output => "
            f"{R_str}\n"
            "note:\n"
            "1. don't add extra contents just structure mentioned output\n"
            "2. If there is mistake in output correct or else keep the original"
            )

    API_URL = "https://router.huggingface.co/v1/chat/completions"
    headers = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}

    def post_request(payload):
        response = requests.post(API_URL, headers=headers, json=payload)
        return response.json()

    payload = {
            "messages": [
                {"role": "user", "content": prompt}
            ],
                "model": "deepseek-ai/DeepSeek-R1:novita"
            }

    response = post_request(payload)
    return response

user_input="Explain Machineee Leaarnin"
user_input = re.sub(r'[^0-9a-zA-Z\s]','',user_input)

response = rag_query(user_input)
print(response)

213
129


NameError: name 'os' is not defined